# EmbedMed

# ------ 

In [22]:
%pip install "python-jose[cryptography]" "passlib[bcrypt]"

   ---------------------------------------- 0.0/525.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/525.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/525.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/525.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/525.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/525.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/525.6 kB ? eta -:--:--
   ------------------- -------------------- 262.1/525.6 kB ? eta -:--:--
   ------------------- -------------------- 262.1/525.6 kB ? eta -:--:--
   ------------------- -------------------- 262.1/525.6 kB ? eta -:--:--
   ------------------- -------------------- 262.1/525.6 kB ? eta -:--:--
   ------------------- -------------------- 262.1/525.6 kB ? eta -:--:--
   ---------------------------------------- 525.6/525.6 kB 240.8 kB/s  0:00:02
   ---------------------------------------- 0.0/3.9 MB ? et

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

from controllers.ProcessController import ProcessController

config = {
    "GROQ_API_KEY": os.getenv("GROQ_API_KEY"),
    "GENERATION_MODEL": "openai/gpt-oss-120b",
    "EMBEDDING_MODEL": "BAAI/bge-small-en-v1.5",
    "VECTOR_DB_PATH": "chroma_db",
}

controller = ProcessController(config)

c:\anaconda\envs\medical_rag\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### loading and chunking

In [3]:
controller.load_pdf(
    pdf_path="assets/hypertension-in-adults-diagnosis-and-management.pdf",
    document_id="NICE-NG136-2026",
    title="Hypertension in Adults: Diagnosis and Management",
    version="NG136",
    publication_date="2019-08-28"
)

controller.chunk_documents(document_id="NICE-NG136-2026")

controller.build_vectorstore(collection_name="hypertension_clinical_kb")

print("Pipeline completed successfully")
print(f"Pages loaded: {len(controller.pages)}")
print(f"Chunks created: {len(controller.chunks)}")

Pipeline completed successfully
Pages loaded: 52
Chunks created: 156


### testing

In [6]:
result = controller.ask("What is the target blood pressure for people with type 2 diabetes?")
print(result["answer"])
print("\nSources:")
for s in result["retrieved_sources"]:
    print(s)

The current recommendation is that people with type 2 diabetes should aim for a clinic blood‑pressure target of **below 140 mm Hg systolic and below 90 mm Hg diastolic** (for adults under 80 years of age)【NICE‑NG136‑2026‑CH‑038 | p. 14】. This target is the same as for people with hypertension with or without type 2 diabetes and replaces earlier suggestions of a lower 130/80 mm Hg target, which were based on limited evidence【NICE‑NG136‑2026‑CH‑113 | p. 38】.  

Educational information only; not a diagnosis or medical advice.

Sources:
{'document_id': 'NICE-NG136-2026', 'page': 38, 'chunk_id': 'NICE-NG136-2026-CH-113', 'preview': 'on people already receiving treatment and that it lacked information on adverse events.  The committee agreed that there was no evidence to suggest that blood pressure targets  sho'}
{'document_id': 'NICE-NG136-2026', 'page': 14, 'chunk_id': 'NICE-NG136-2026-CH-038', 'preview': 'hypertension in pregnancy.  See also table 1 for clinic blood pressure targets for p

### 20 questions evaluation and displaying chnk info

In [7]:
import sys
sys.path.append("evaluation")

from evaluation.eval_questions import EVAL_QUESTIONS
from evaluation.run_evaluation import run_retrieval, compare_k_values

sample_results = run_retrieval(controller, EVAL_QUESTIONS, k=5)

for r in sample_results:
    print("Question:", r["question"])
    for chunk in r["retrieved_chunks"]:
        print(f"  [{chunk['chunk_id']}] score={chunk['score']} page={chunk['page']}")
        print(f"  {chunk['chunk_text'][:100]}...")
    print()

Question: What is the target blood pressure for adults under 80 without diabetes?
  [NICE-NG136-2026-CH-038] score=0.8348 page=14
  hypertension in pregnancy. 
See also table 1 for clinic blood pressure targets for people aged under...
  [NICE-NG136-2026-CH-044] score=0.8283 page=16
  below 140/90 mmHg and ensure that it is maintained below that level. See also 
table 1 for guidance ...
  [NICE-NG136-2026-CH-115] score=0.8246 page=39
  people without hypertension. They also had concerns about the relevance of the study 
design. The co...
  [NICE-NG136-2026-CH-045] score=0.799 page=16
  hypertension, use the average blood pressure level taken during the person's 
usual waking hours (se...
  [NICE-NG136-2026-CH-119] score=0.7949 page=40
  Blood pressure targets for people with cardiovascular disease 
Recommendation 1.4.23 
Why the commit...

Question: What is the target blood pressure for adults aged 80 and over?
  [NICE-NG136-2026-CH-044] score=0.8509 page=16
  below 140/90 mmHg and ens

# comparing 3 diffirent ks

In [8]:
sample_questions = EVAL_QUESTIONS[:3]
k_comparison = compare_k_values(controller, sample_questions)

for question, k_results in k_comparison.items():
    print("=" * 80)
    print("Question:", question)
    for k, chunks in k_results.items():
        print(f"\n  --- k={k} ({len(chunks)} results) ---")
        for c in chunks:
            print(f"  [{c['chunk_id']}] page={c['page']} score={c['score']}")

Question: What is the target blood pressure for adults under 80 without diabetes?

  --- k=3 (3 results) ---
  [NICE-NG136-2026-CH-038] page=14 score=0.8348
  [NICE-NG136-2026-CH-044] page=16 score=0.8283
  [NICE-NG136-2026-CH-115] page=39 score=0.8246

  --- k=5 (5 results) ---
  [NICE-NG136-2026-CH-038] page=14 score=0.8348
  [NICE-NG136-2026-CH-044] page=16 score=0.8283
  [NICE-NG136-2026-CH-115] page=39 score=0.8246
  [NICE-NG136-2026-CH-045] page=16 score=0.799
  [NICE-NG136-2026-CH-119] page=40 score=0.7949

  --- k=10 (10 results) ---
  [NICE-NG136-2026-CH-038] page=14 score=0.8348
  [NICE-NG136-2026-CH-044] page=16 score=0.8283
  [NICE-NG136-2026-CH-115] page=39 score=0.8246
  [NICE-NG136-2026-CH-045] page=16 score=0.799
  [NICE-NG136-2026-CH-119] page=40 score=0.7949
  [NICE-NG136-2026-CH-113] page=38 score=0.7912
  [NICE-NG136-2026-CH-146] page=48 score=0.788
  [NICE-NG136-2026-CH-077] page=28 score=0.7872
  [NICE-NG136-2026-CH-155] page=52 score=0.7852
  [NICE-NG136-2026-CH-

### Compare 2 chunks configs

In [9]:
controller_small = ProcessController(config)

# 2. نحمل نفس الملف (مع إضافة البارامترز المطلوبة)
controller_small.load_pdf(
    pdf_path="assets/hypertension-in-adults-diagnosis-and-management.pdf",
    document_id="NICE-NG136-2026",
    title="Hypertension in Adults: Diagnosis and Management",
    version="NG136",
    publication_date="2019-08-28"
)

# 3. هنجرب Chunk size أصغر (مثلاً 500)
controller_small.chunk_documents(document_id="NICE-NG136-2026", chunk_size=500, chunk_overlap=50)

# 4. نعمل Collection جديدة في قاعدة البيانات باسم مختلف عشان متدخلش في القديمة
controller_small.build_vectorstore(collection_name="hypertension_small_chunks")

print(f"Original Chunks count: {len(controller.chunks)}")
print(f"Small Chunks count: {len(controller_small.chunks)}")

# 5. نقارن نتيجة نفس السؤال بين الإعدادين
test_q = "What is the target blood pressure for people with type 2 diabetes?"

print("\n=== Results from ORIGINAL chunks ===")
res1 = controller.ask(test_q)
for s in res1["retrieved_sources"][:2]: 
    # حولناها لـ String الأول عشان نقدر نقص منها أول 150 حرف
    print("-", str(s)[:150], "...")

print("\n=== Results from SMALL chunks ===")
res2 = controller_small.ask(test_q)
for s in res2["retrieved_sources"][:2]:
    # حولناها لـ String الأول
    print("-", str(s)[:150], "...")

Original Chunks count: 156
Small Chunks count: 240

=== Results from ORIGINAL chunks ===
- {'document_id': 'NICE-NG136-2026', 'page': 38, 'chunk_id': 'NICE-NG136-2026-CH-113', 'preview': 'on people already receiving treatment and that it lac ...
- {'document_id': 'NICE-NG136-2026', 'page': 14, 'chunk_id': 'NICE-NG136-2026-CH-038', 'preview': 'hypertension in pregnancy.  See also table 1 for clin ...

=== Results from SMALL chunks ===
- {'document_id': 'NICE-NG136-2026', 'page': 38, 'chunk_id': 'NICE-NG136-2026-CH-171', 'preview': 'The committee agreed that there was no evidence to su ...
- {'document_id': 'NICE-NG136-2026', 'page': 38, 'chunk_id': 'NICE-NG136-2026-CH-166', 'preview': 'targets should be used. The committee agreed that in  ...


### labeling and calculation of percision3,5 and avg

In [10]:
sample_questions = EVAL_QUESTIONS[:3]
eval_results = run_retrieval(controller, sample_questions, k=5)

for r in eval_results:
    print("=" * 100)
    print(f"Question: {r['question']}")
    print("=" * 100)
    
    for i, chunk in enumerate(r["retrieved_chunks"], 1):
        print(f"\n--- Chunk {i} | ID: [{chunk['chunk_id']}] | Page: {chunk['page']} ---")
        print(chunk['chunk_text'])
        print("-" * 50)

Question: What is the target blood pressure for adults under 80 without diabetes?

--- Chunk 1 | ID: [NICE-NG136-2026-CH-038] | Page: 14 ---
hypertension in pregnancy. 
See also table 1 for clinic blood pressure targets for people aged under 80 and table 2 for 
clinic blood pressure targets for people aged 80 and over. The tables cover people with 
hypertension (with or without type 2 diabetes) as well as people with chronic kidney 
disease or type 1 diabetes. 
Table 1: Clinic blood pressure targets for people aged under 80 
Person under 80 with: 
Clinic blood 
pressure 
target 
Source 
• hypertension (with or without type 2 
diabetes) or 
• type 1 diabetes plus albumin to 
creatinine ratio less than 70 mg/mmol 
or 
• chronic kidney disease plus albumin 
to creatinine ratio less than 70 mg/
mmol 
Below 
140/90 
Recommendation 1.4.20 
NICE's guideline on type 1 
diabetes in adults 
(recommendation 1.13.8) 
NICE's guideline on chronic kidney 
disease (recommendation 1.6.1)
--------------

In [12]:
%pip install pandas
import pandas as pd

# Task 5: Manually label chunks (1 = Relevant, 0 = Not Relevant)
manual_labels = {
    "Q1: Target BP under 80 without diabetes": [1, 1, 1, 1, 0],
    "Q2: Target BP adults aged 80 and over":   [1, 1, 0, 0, 0],
    "Q3: Target BP people with type 2 diabetes":[0, 1, 0, 0, 1] 
}

# Task 6: Calculate Precision@3 and Precision@5, plus averages
def calc_precision_at_k(labels, k):
    return sum(labels[:k]) / k

results_data = []

for question, labels in manual_labels.items():
    p3 = calc_precision_at_k(labels, 3)
    p5 = calc_precision_at_k(labels, 5)
    results_data.append({
        "Question": question,
        "P@3": round(p3, 2),
        "P@5": round(p5, 2)
    })

eval_df = pd.DataFrame(results_data)

print("=== Evaluation Metrics ===")
print(eval_df.to_string(index=False))

print("\n=== Averages ===")
print(f"Average Precision@3: {eval_df['P@3'].mean():.2f}")
print(f"Average Precision@5: {eval_df['P@5'].mean():.2f}")

   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
    --------------------------------------- 0.3/11.3 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.3 MB 1.9 MB/s eta 0:00:06
   --- ------------------------------------ 1.0/11.3 MB 1.9 MB/s eta 0:00:06
   ---- ----------------------------------- 1.3/11.3 MB 1.9 MB/s eta 0:00:06
   ------ --------------------------------- 1.8/11.3 MB 1.9 MB/s eta 0:00:06
   ------- -------------------------------- 2.1/11.3 MB 1.9 MB/s eta 0:00:05
   --------- ------------------------------ 2.6/11.3 MB 1.9 MB/s eta 0:00:05
   ---------- ----------------------------- 2.9/11.3 MB 1.9 MB/s eta 0:00:05
   ------------ --------------------------- 3.4/11.3 MB 1.9 MB/s eta 0:00:05
   ------------ --------------------------- 3.7/11.3 MB 1.9 MB/s eta 0:00:05
   -------------- ------------------------- 4.2/11.3 MB 1.9 MB/s eta 0:00:04
   --------------- ------------------------ 4.5/11.3 MB 1.9 MB/s eta 0:00:04
   ----------

### Failure Case

#### Task 7: Retrieval Failure Case Analysis

* **Question:** *"What is the target blood pressure for people with type 2 diabetes?"*
* **Failure Mode:** `False Positive Distractor` & `Semantic Drift`
* **Observed Behavior:** The retriever assigned the highest score (Rank 1) to Chunk `[NICE-NG136-2026-CH-113]`. While this chunk perfectly matched the keywords ("type 2 diabetes", "blood pressure targets"), it was actually a "Rationale and Discussion" section explaining the *lack of evidence* for different targets, rather than stating the current actionable target. The actual correct answer was pushed to Rank 2 and Rank 5.
* **Root Cause:** The embedding model successfully captured semantic similarity based on medical terminology but failed to distinguish between the **context of a medical recommendation** versus the **context of a committee discussion/historical evidence**. It couldn't grasp the "intent" of finding a specific numerical target over general discourse.

### keyword search

In [13]:
# Task 8 (Optional): Try Keyword Search on the failure case question
test_question = "What is the target blood pressure for people with type 2 diabetes?"
keywords = ["target", "type 2 diabetes", "0"] # ضفنا الرقم ككلمة مفتاحية لنرى مدى دقتها
140/9
print(f"=== Keyword Search Results ===\n")
print(f"Question: {test_question}\n")

keyword_results = []

# البحث المباشر بالكلمات في كل الـ Chunks الموجودة في الذاكرة
for chunk in controller.chunks:
    text_lower = chunk.page_content.lower()
    # التأكد أن كل الكلمات المفتاحية موجودة في القطعة
    if "type 2 diabetes" in text_lower and "target" in text_lower:
        keyword_results.append(chunk)

# طباعة أول 3 نتائج
if not keyword_results: 
    print("No chunks found with these exact keywords.")
else:
    for i, chunk in enumerate(keyword_results[:3], 1):
        print(f"--- Keyword Result {i} | ID: [{chunk.metadata.get('chunk_id')}] | Page: {chunk.metadata.get('page_number')} ---")
        print(chunk.page_content[:300].replace('\n', ' '))
        print("-" * 80)

=== Keyword Search Results ===

Question: What is the target blood pressure for people with type 2 diabetes?

--- Keyword Result 1 | ID: [NICE-NG136-2026-CH-038] | Page: 14 ---
hypertension in pregnancy.  See also table 1 for clinic blood pressure targets for people aged under 80 and table 2 for  clinic blood pressure targets for people aged 80 and over. The tables cover people with  hypertension (with or without type 2 diabetes) as well as people with chronic kidney  dise
--------------------------------------------------------------------------------
--- Keyword Result 2 | ID: [NICE-NG136-2026-CH-041] | Page: 15 ---
1.4.16 Check for postural hypotension (see recommendation 1.1.5) in people with  hypertension and:  • type 2 diabetes or  • symptoms of postural hypotension (see also recommendation 1.1.7) or  • aged 80 and over.  In people with a significant postural drop or symptoms of postural  hypotension, treat
-------------------------------------------------------------------------

### Task 9: Final Retrieval Configuration

Based on the evaluation of multiple queries across different configurations, the optimal retrieval setup for this medical clinical guideline (NICE NG136) is as follows:

* **Chunking Strategy:** A moderate chunk size (e.g., ~1000 characters) with a small overlap (e.g., 100 characters). This preserves the complete context of multi-part medical guidelines and tables without fragmenting crucial clinical targets.
* **Top-K Value:** **k=5**. Our evaluation showed that while $k=3$ works for highly specific questions (P@3 = 0.67 average), complex or deeply nested answers often surface at ranks 4 or 5. Setting $k=5$ ensures high recall without flooding the LLM context window.
* **Retrieval Method:** **Hybrid Search**. The pure dense vector retrieval (`BAAI/bge-small-en-v1.5`) suffered from semantic drift (e.g., retrieving historical committee rationales instead of actual targets for Type 2 Diabetes). Combining dense embeddings with sparse keyword search (BM25) would effectively anchor the semantic search to exact medical terminology and numerical targets.

## Day-3

In [ ]:
"""
Day 3 — Grounded Answer Layer for EmbedMed-RAG
Sits on top of the existing ProcessController (Day 1/2) and adds:
  - a citation-bound system prompt
  - JSON-schema-enforced output
  - insufficient-evidence refusal (score-threshold based)
  - patient-specific safety refusal (regex/keyword gate, runs BEFORE retrieval)
  - a manual verification hook so every citation can be traced to the exact chunk text

Uses the same Interface + Provider + Factory pattern already in stores/llm and
stores/vectordb — this file only orchestrates, it doesn't reimplement retrieval.
"""

from pathlib import Path
from jsonschema import validate, ValidationError
import json



# النوت بوك بتاعك شغال جوه فولدر src، فهندخل على schema مباشرة
SCHEMA_PATH = Path.cwd() / "schema" / "response_schema.json"

with open(SCHEMA_PATH, encoding="utf-8") as f:
    RESPONSE_SCHEMA = json.load(f)

print("✅ Schema loaded successfully!")


GROUNDING_SYSTEM_PROMPT = """You are a citation-bound clinical evidence assistant for EmbedMed-RAG.
Your only source of truth is the NICE NG136 (Hypertension in Adults) context passages
provided below. You are NOT a general medical advisor and you have NO knowledge beyond
what is in the context.

RULES — follow every one exactly:
1. Answer ONLY using the context passages given. Never draw on outside medical training.
2. Every sentence in "recommendation" must be directly traceable to the text in "evidence".
3. Return ONLY a JSON object matching this exact structure, nothing else:
   {
     "recommendation": "...",
     "evidence": "...",
     "citations": [{"document_id": "...", "section": "...", "page_number": N, "chunk_id": "..."}],
     "confidence": "high" | "medium" | "low" | "insufficient",
     "flags": []
   }
4. If the retrieved context does not clearly answer the question, set confidence to
   "insufficient", leave evidence/citations empty, and write a plain refusal in
   "recommendation" — never soften a refusal into a partial guess.
5. Never invent a page number, section title, or chunk ID. Copy them exactly from the
   context block metadata you were given.
6. You must never produce individualized dosing, titration schedules, or a treatment
   plan addressed to "the patient" — NG136 is a guideline, not a prescription. Flag
   any such request instead of answering it (see patient-specific gate below).
"""


# ---------------------------------------------------------------------------
# 2. Patient-specific safety gate — runs BEFORE retrieval, cheap keyword screen.
#    Not meant to be exhaustive NLP; it's a first-line trip-wire your team can
#    tighten during Day 4 based on what slips through in testing.
# ---------------------------------------------------------------------------
PATIENT_SPECIFIC_PATTERNS = [
    r"\bmy (blood pressure|bp|dose|dosage|medication)\b",
    r"\bshould i (take|stop|increase|decrease)\b",
    r"\bhow much (should i|do i need)\b",
    r"\bi am (a |on )?\d+\s*(years old|yo|mg)\b",
    r"\bfor (my|a) patient (who|with)\b.*\bmg\b",
]

def is_patient_specific(question: str) -> bool:
    q = question.lower()
    return any(re.search(p, q) for p in PATIENT_SPECIFIC_PATTERNS)


def refusal_payload(reason: str, flag: str) -> dict:
    return {
        "recommendation": reason,
        "evidence": "",
        "citations": [],
        "confidence": "insufficient",
        "flags": [flag],
    }


# ---------------------------------------------------------------------------
# 3. Prompt assembly — pulls document_id / section / page / chunk_id straight
#    from the metadata your ingest step already attaches (Day 1).
# ---------------------------------------------------------------------------
def build_prompt(question: str, retrieved) -> str:
    """retrieved: list[(doc, score)] as returned by ChromaProvider.search_with_scores"""
    context_blocks = []
    for doc, score in retrieved:
        meta = doc.metadata
        context_blocks.append(
            f"[document_id={meta.get('document_id')} | section={meta.get('section', 'unspecified')} "
            f"| page={meta.get('page_number')} | chunk_id={meta.get('chunk_id', meta.get('id'))} "
            f"| score={score:.3f}]\n{doc.page_content}"
        )
    context = "\n\n".join(context_blocks)

    return f"""{GROUNDING_SYSTEM_PROMPT}

Context:
{context}

Question: {question}

Respond with ONLY the JSON object described above.
"""


# ---------------------------------------------------------------------------
# 4. Main entry point
# ---------------------------------------------------------------------------
class GroundedAnswerController:
    """
    Wraps an existing 
     instance (Day 1/2) — reuses its
    vectorstore + llm_provider, adds the grounding/refusal/validation layer.
    """

    def __init__(self, process_controller, confidence_threshold: float = 0.3, top_k: int = 5):
        self.pc = process_controller  # your existing ProcessController
        self.threshold = confidence_threshold
        self.top_k = top_k

    def ask(self, question: str) -> dict:
        # --- Gate 1: patient-specific safety refusal (before any retrieval) ---
        if is_patient_specific(question):
            return refusal_payload(
                "I can't provide individualized dosing or treatment decisions. "
                "NG136 gives population-level guidance only — please direct this "
                "question to a qualified clinician who knows the patient's full history.",
                flag="patient_specific_blocked",
            )

        # --- Retrieve using the existing vectorstore/provider (Day 1/2 code) ---
        retrieved = self.pc.vectordb_provider.search_with_scores(question, k=self.top_k)
        top_score = retrieved[0][1] if retrieved else -999.0

        # --- Gate 2: insufficient-evidence refusal ---
        if not retrieved or top_score < self.threshold:
            return refusal_payload(
                "I couldn't find strong enough evidence in NG136 to answer this "
                "confidently. This may be out of scope for this guideline, or the "
                "wording may need to be more specific — try rephrasing.",
                flag="insufficient_evidence",
            )

        # --- Generate ---
        prompt = build_prompt(question, retrieved)
        raw = self.pc.llm_provider.generate(prompt)  # existing GroqProvider.generate()
        answer = self._parse_and_validate(raw)
        return answer

    @staticmethod
    def _parse_and_validate(raw: str) -> dict:
        # strip stray markdown fences some models still add despite the prompt
        cleaned = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        try:
            parsed = json.loads(cleaned)
        except json.JSONDecodeError as e:
            return refusal_payload(
                f"Generation did not return valid JSON ({e}). Treating as insufficient "
                "rather than guessing at a fix.",
                flag="malformed_output",
            )
        try:
            validate(instance=parsed, schema=RESPONSE_SCHEMA)
        except ValidationError as e:
            return refusal_payload(
                f"Model output failed schema validation ({e.message}). Discarding "
                "rather than showing an unverified answer.",
                flag="schema_validation_failed",
            )
        return parsed


# ---------------------------------------------------------------------------
# 5. Manual citation-verification helper — required for the "walk a reviewer
#    from claim -> citation -> literal retrieved text" demo.
# ---------------------------------------------------------------------------
def verify_citation(answer: dict, retrieved) -> list[dict]:
    """
    For each citation in `answer`, find the matching retrieved chunk and return
    it side-by-side so a human can eyeball whether evidence text actually
    appears in the chunk. This does NOT auto-approve anything - it's a lookup,
    the judgment call stays manual per the Day 3 requirement.
    """
    lookup = {doc.metadata.get("chunk_id", doc.metadata.get("id")): doc for doc, _ in retrieved}
    report = []
    for c in answer.get("citations", []):
        chunk = lookup.get(c["chunk_id"])
        report.append({
            "citation": c,
            "chunk_found": chunk is not None,
            "chunk_text": chunk.page_content if chunk else None,
            "evidence_substring_present": (
                answer["evidence"].strip()[:40].lower() in chunk.page_content.lower()
                if chunk else False
            ),
        })
    return report

✅ Schema loaded successfully!


In [16]:
"""
Day 3 — Grounded Answer Layer for EmbedMed-RAG
Sits on top of the existing ProcessController (Day 1/2) and adds:
  - a citation-bound system prompt
  - JSON-schema-enforced output
  - insufficient-evidence refusal (score-threshold based)
  - patient-specific safety refusal (regex/keyword gate, runs BEFORE retrieval)
  - a manual verification hook so every citation can be traced to the exact chunk text

Uses the same Interface + Provider + Factory pattern already in stores/llm and
stores/vectordb — this file only orchestrates, it doesn't reimplement retrieval.
"""

from pathlib import Path
from jsonschema import validate, ValidationError
import json



GROUNDING_SYSTEM_PROMPT = """You are a citation-bound clinical evidence assistant for EmbedMed-RAG.
Your only source of truth is the NICE NG136 (Hypertension in Adults) context passages
provided below. You are NOT a general medical advisor and you have NO knowledge beyond
what is in the context.

RULES — follow every one exactly:
1. Answer ONLY using the context passages given. Never draw on outside medical training.
2. Every sentence in "recommendation" must be directly traceable to the text in "evidence".
3. Return ONLY a JSON object matching this exact structure, nothing else:
   {
     "recommendation": "...",
     "evidence": "...",
     "citations": [{"document_id": "...", "section": "...", "page_number": N, "chunk_id": "..."}],
     "confidence": "high" | "medium" | "low" | "insufficient",
     "flags": []
   }
4. If the retrieved context does not clearly answer the question, set confidence to
   "insufficient", leave evidence/citations empty, and write a plain refusal in
   "recommendation" — never soften a refusal into a partial guess.
5. Never invent a page number, section title, or chunk ID. Copy them exactly from the
   context block metadata you were given.
6. You must never produce individualized dosing, titration schedules, or a treatment
   plan addressed to "the patient" — NG136 is a guideline, not a prescription. Flag
   any such request instead of answering it (see patient-specific gate below).
"""


# ---------------------------------------------------------------------------
# 2. Patient-specific safety gate — runs BEFORE retrieval, cheap keyword screen.
#    Not meant to be exhaustive NLP; it's a first-line trip-wire your team can
#    tighten during Day 4 based on what slips through in testing.
# ---------------------------------------------------------------------------
PATIENT_SPECIFIC_PATTERNS = [
    r"\bmy (blood pressure|bp|dose|dosage|medication)\b",
    r"\bshould i (take|stop|increase|decrease)\b",
    r"\bhow much (should i|do i need)\b",
    r"\bi am (a |on )?\d+\s*(years old|yo|mg)\b",
    r"\bfor (my|a) patient (who|with)\b.*\bmg\b",
]

def is_patient_specific(question: str) -> bool:
    q = question.lower()
    return any(re.search(p, q) for p in PATIENT_SPECIFIC_PATTERNS)


def refusal_payload(reason: str, flag: str) -> dict:
    return {
        "recommendation": reason,
        "evidence": "",
        "citations": [],
        "confidence": "insufficient",
        "flags": [flag],
    }


# ---------------------------------------------------------------------------
# 3. Prompt assembly — pulls document_id / section / page / chunk_id straight
#    from the metadata your ingest step already attaches (Day 1).
# ---------------------------------------------------------------------------
def build_prompt(question: str, retrieved) -> str:
    """retrieved: list[(doc, score)] as returned by ChromaProvider.search_with_scores"""
    context_blocks = []
    for doc, score in retrieved:
        meta = doc.metadata
        context_blocks.append(
            f"[document_id={meta.get('document_id')} | section={meta.get('section', 'unspecified')} "
            f"| page={meta.get('page_number')} | chunk_id={meta.get('chunk_id', meta.get('id'))} "
            f"| score={score:.3f}]\n{doc.page_content}"
        )
    context = "\n\n".join(context_blocks)

    return f"""{GROUNDING_SYSTEM_PROMPT}

Context:
{context}

Question: {question}

Respond with ONLY the JSON object described above.
"""


# ---------------------------------------------------------------------------
# 4. Main entry point
# ---------------------------------------------------------------------------
class GroundedAnswerController:
    """
    Wraps an existing ProcessController instance (Day 1/2) — reuses its
    vectorstore + llm_provider, adds the grounding/refusal/validation layer.
    """

    def __init__(self, process_controller, confidence_threshold: float = 0.3, top_k: int = 5):
        self.pc = process_controller  # your existing ProcessController
        self.threshold = confidence_threshold
        self.top_k = top_k

    def ask(self, question: str) -> dict:
        # --- Gate 1: patient-specific safety refusal (before any retrieval) ---
        if is_patient_specific(question):
            return refusal_payload(
                "I can't provide individualized dosing or treatment decisions. "
                "NG136 gives population-level guidance only — please direct this "
                "question to a qualified clinician who knows the patient's full history.",
                flag="patient_specific_blocked",
            )

        # --- Retrieve using the existing vectorstore/provider (Day 1/2 code) ---
        retrieved = self.pc.vectordb_provider.search_with_scores(question, k=self.top_k)
        top_score = retrieved[0][1] if retrieved else -999.0

        # --- Gate 2: insufficient-evidence refusal ---
        if not retrieved or top_score < self.threshold:
            return refusal_payload(
                "I couldn't find strong enough evidence in NG136 to answer this "
                "confidently. This may be out of scope for this guideline, or the "
                "wording may need to be more specific — try rephrasing.",
                flag="insufficient_evidence",
            )

        # --- Generate ---
        prompt = build_prompt(question, retrieved)
        raw = self.pc.llm_provider.generate(prompt)  # existing GroqProvider.generate()
        answer = self._parse_and_validate(raw)
        return answer

    @staticmethod
    def _parse_and_validate(raw: str) -> dict:
        # strip stray markdown fences some models still add despite the prompt
        cleaned = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        try:
            parsed = json.loads(cleaned)
        except json.JSONDecodeError as e:
            return refusal_payload(
                f"Generation did not return valid JSON ({e}). Treating as insufficient "
                "rather than guessing at a fix.",
                flag="malformed_output",
            )
        try:
            validate(instance=parsed, schema=RESPONSE_SCHEMA)
        except ValidationError as e:
            return refusal_payload(
                f"Model output failed schema validation ({e.message}). Discarding "
                "rather than showing an unverified answer.",
                flag="schema_validation_failed",
            )
        return parsed


# ---------------------------------------------------------------------------
# 5. Manual citation-verification helper — required for the "walk a reviewer
#    from claim -> citation -> literal retrieved text" demo.
# ---------------------------------------------------------------------------
def verify_citation(answer: dict, retrieved) -> list[dict]:
    """
    For each citation in `answer`, find the matching retrieved chunk and return
    it side-by-side so a human can eyeball whether evidence text actually
    appears in the chunk. This does NOT auto-approve anything - it's a lookup,
    the judgment call stays manual per the Day 3 requirement.
    """
    lookup = {doc.metadata.get("chunk_id", doc.metadata.get("id")): doc for doc, _ in retrieved}
    report = []
    for c in answer.get("citations", []):
        chunk = lookup.get(c["chunk_id"])
        report.append({
            "citation": c,
            "chunk_found": chunk is not None,
            "chunk_text": chunk.page_content if chunk else None,
            "evidence_substring_present": (
                answer["evidence"].strip()[:40].lower() in chunk.page_content.lower()
                if chunk else False
            ),
        })
    return report

In [20]:
import json
import re
from pathlib import Path
from jsonschema import validate, ValidationError

# ==========================================
# 1. إعدادات الـ Schema والـ Prompts
# ==========================================

SCHEMA_PATH = Path.cwd() / "schema" / "response_schema.json"
with open(SCHEMA_PATH, encoding="utf-8") as f:
    RESPONSE_SCHEMA = json.load(f)

GROUNDING_SYSTEM_PROMPT = """You are a citation-bound clinical evidence assistant for EmbedMed-RAG.
Your only source of truth is the NICE NG136 (Hypertension in Adults) context passages
provided below. You are NOT a general medical advisor and you have NO knowledge beyond
what is in the context.

RULES — follow every one exactly:
1. Answer ONLY using the context passages given. Never draw on outside medical training.
2. Every sentence in "recommendation" must be directly traceable to the text in "evidence".
3. Return ONLY a JSON object matching this exact structure, nothing else:
   {
     "recommendation": "...",
     "evidence": "...",
     "citations": [{"document_id": "...", "section": "...", "page_number": N, "chunk_id": "..."}],
     "confidence": "high" | "medium" | "low" | "insufficient",
     "flags": []
   }
4. If the retrieved context does not clearly answer the question, set confidence to
   "insufficient", leave evidence/citations empty, and write a plain refusal in
   "recommendation" — never soften a refusal into a partial guess.
5. Never invent a page number, section title, or chunk ID. Copy them exactly from the
   context block metadata you were given.
6. You must never produce individualized dosing, titration schedules, or a treatment
   plan addressed to "the patient" — NG136 is a guideline, not a prescription. Flag
   any such request instead of answering it (see patient-specific gate below).
"""

PATIENT_SPECIFIC_PATTERNS = [
    r"\bmy (blood pressure|bp|dose|dosage|medication)\b",
    r"\bshould i (take|stop|increase|decrease)\b",
    r"\bhow much (should i|do i need)\b",
    r"\bi am (a |on )?\d+\s*(years old|yo|mg)\b",
    r"\bfor (my|a) patient (who|with)\b.*\bmg\b",
]

def is_patient_specific(question: str) -> bool:
    q = question.lower()
    return any(re.search(p, q) for p in PATIENT_SPECIFIC_PATTERNS)

def refusal_payload(reason: str, flag: str) -> dict:
    return {
        "recommendation": reason,
        "evidence": "",
        "citations": [],
        "confidence": "insufficient",
        "flags": [flag],
    }

def build_prompt(question: str, retrieved) -> str:
    context_blocks = []
    for doc, score in retrieved:
        meta = doc.metadata
        context_blocks.append(
            f"[document_id={meta.get('document_id')} | section={meta.get('section', 'unspecified')} "
            f"| page={meta.get('page_number')} | chunk_id={meta.get('chunk_id', meta.get('id'))} "
            f"| score={score:.3f}]\n{doc.page_content}"
        )
    context = "\n\n".join(context_blocks)
    return f"{GROUNDING_SYSTEM_PROMPT}\n\nContext:\n{context}\n\nQuestion: {question}\n\nRespond with ONLY the JSON object described above."

# ==========================================
# 2. المتحكم الرئيسي بعد التعديل الذكي للـ LLM
# ==========================================
class GroundedAnswerController:
    def __init__(self, process_controller, confidence_threshold: float = 0.3, top_k: int = 5):
        self.pc = process_controller  
        self.threshold = confidence_threshold
        self.top_k = top_k

    def _generate_llm_response(self, prompt: str) -> str:
        """دالة ذكية بتدور على الاسم الصح لـ GroqProvider عشان تشغله"""
        llm = self.pc.llm
        if hasattr(llm, 'invoke'):
            res = llm.invoke(prompt)
            return res.content if hasattr(res, 'content') else str(res)
        elif hasattr(llm, 'generate'):
            return llm.generate(prompt)
        elif hasattr(llm, 'generate_text'):
            return llm.generate_text(prompt)
        elif hasattr(llm, '__call__'):
            return llm(prompt)
        else:
            raise AttributeError("مش قادر ألاقي الدالة المناسبة لتشغيل GroqProvider. يرجى مراجعة ملف ProcessController.py")

    def ask(self, question: str) -> dict:
        # --- Gate 1: Patient-specific safety ---
        if is_patient_specific(question):
            return refusal_payload(
                "I can't provide individualized dosing or treatment decisions. NG136 gives population-level guidance only.",
                flag="patient_specific_blocked",
            )

        # --- Retrieve Context ---
        retrieved = self.pc.vectordb.search_with_scores(question, k=self.top_k)
        top_score = retrieved[0][1] if retrieved else -999.0

        # --- Gate 2: Insufficient evidence ---
        if not retrieved or top_score < self.threshold:
            return refusal_payload(
                "I couldn't find strong enough evidence in NG136 to answer this confidently.",
                flag="insufficient_evidence",
            )

        # --- Generate Answer ---
        prompt = build_prompt(question, retrieved)
        raw = self._generate_llm_response(prompt) # استخدمنا الدالة الذكية هنا 
        answer = self._parse_and_validate(raw)
        return answer

    @staticmethod
    def _parse_and_validate(raw: str) -> dict:
        cleaned = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        try:
            parsed = json.loads(cleaned)
        except json.JSONDecodeError as e:
            return refusal_payload(f"Generation did not return valid JSON ({e}).", flag="malformed_output")
        try:
            validate(instance=parsed, schema=RESPONSE_SCHEMA)
        except ValidationError as e:
            return refusal_payload(f"Model output failed schema validation ({e.message}).", flag="schema_validation_failed")
        return parsed


def verify_citation(answer: dict, retrieved) -> list[dict]:
    lookup = {doc.metadata.get("chunk_id", doc.metadata.get("id")): doc for doc, _ in retrieved}
    report = []
    for c in answer.get("citations", []):
        chunk = lookup.get(c["chunk_id"])
        report.append({
            "citation": c,
            "chunk_found": chunk is not None,
            "chunk_text": chunk.page_content if chunk else None,
            "evidence_substring_present": (
                answer["evidence"].strip()[:40].lower() in chunk.page_content.lower()
                if chunk else False
            ),
        })
    return report

# ==========================================
# 3. تشغيل الاختبارات واستخراج النتائج
# ==========================================

grounded_controller = GroundedAnswerController(
    process_controller=controller, 
    confidence_threshold=0.4, 
    top_k=5
)

print("🚀 Starting Day 3 Deliverables Extraction...\n")

# ---------------------------------------------------------
# 1. Supported Question
# ---------------------------------------------------------
q_supported = "What is the target clinic blood pressure for adults aged under 80 years?"
ans_supported = grounded_controller.ask(q_supported)
print("✅ TEST 1: Supported Question")
print(f"Q: {q_supported}")
print(json.dumps(ans_supported, indent=2))

print("\n🔍 VERIFICATION REPORT (For the Demo):")
retrieved_docs_for_demo = grounded_controller.pc.vectordb.search_with_scores(q_supported, k=5)
verification = verify_citation(ans_supported, retrieved_docs_for_demo)
print(json.dumps(verification, indent=2))
print("-" * 50)

# ---------------------------------------------------------
# 2. Unsupported Question
# ---------------------------------------------------------
q_unsupported = "What is the recommended treatment for a torn ACL or broken leg?"
ans_unsupported = grounded_controller.ask(q_unsupported)
print("\n⚠️ TEST 2: Unsupported Question (Should trigger Insufficient Evidence)")
print(f"Q: {q_unsupported}")
print(json.dumps(ans_unsupported, indent=2))
print("-" * 50)

# ---------------------------------------------------------
# 3. Unsafe Question
# ---------------------------------------------------------
q_unsafe = "I am 55 years old and my blood pressure is 150/95. What dosage of Amlodipine should I take?"
ans_unsafe = grounded_controller.ask(q_unsafe)
print("\n🛑 TEST 3: Unsafe Question (Should trigger Patient-Specific Refusal)")
print(f"Q: {q_unsafe}")
print(json.dumps(ans_unsafe, indent=2))
print("-" * 50)

🚀 Starting Day 3 Deliverables Extraction...

✅ TEST 1: Supported Question
Q: What is the target clinic blood pressure for adults aged under 80 years?
{
  "recommendation": "For adults aged under 80 years, the guideline recommends a clinic blood pressure target of below 140/90\u202fmmHg.",
  "evidence": "The guideline states that clinic blood pressure should be reduced to \"below 140/90 mmHg and ensure that it is maintained below that level\" for adults under 80 (see Table\u202f1) and Table\u202f1 lists the target for people under 80 as \"Below 140/90\".",
  "citations": [
    {
      "document_id": "NICE-NG136-2026",
      "section": "unspecified",
      "page_number": 16,
      "chunk_id": "NICE-NG136-2026-CH-044"
    },
    {
      "document_id": "NICE-NG136-2026",
      "section": "unspecified",
      "page_number": 14,
      "chunk_id": "NICE-NG136-2026-CH-038"
    }
  ],
  "confidence": "high",
  "flags": []
}

🔍 VERIFICATION REPORT (For the Demo):
[
  {
    "citation": {
      "d